<a href="https://colab.research.google.com/github/saathvikMD/dqn/blob/main/frozen_lake_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gym
import keras
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class Engine():
  def __init__(self, max_memory, env_name, lr):
    self.lr = lr
    self.memory = []
    self.discount = 0.9
    self.max_memory = max_memory
    self.env_name = env_name
    self.env = gym.make(self.env_name)
    self.epsilon = 0.99
    self.epsilon_decay_rate = 0.05
    self.epsilon_deacy_rate_decay = 0.009
    self.min_epsilon = 0.8
    self.input_shape = 1
    self.output_shape = int(self.env.action_space.n)
    self.model = keras.models.Sequential()
    self.model.add(keras.layers.Dense(self.input_shape, input_shape = (1, self.input_shape)))
    self.model.add(keras.layers.Dense(self.input_shape * 3))
    self.model.add(keras.layers.Dense(self.output_shape * 2))
    self.model.add(keras.layers.Dense(self.output_shape))
    self.model.compile(loss = 'mean_squared_error', optimizer = keras.optimizers.Adam(learning_rate=self.lr))
  
  def remember(self, current_state, action, reward, next_state, game_over):
      transition = [current_state, action, reward, next_state]
      self.memory.append([transition, game_over])

  def get_batch(self, batch_size, model):
      len_memory = len(self.memory)
      num_inputs = self.input_shape
      num_outputs = self.output_shape

      inputs = np.zeros((min(batch_size, len_memory), num_inputs))
      targets = np.zeros((min(batch_size, len_memory), num_outputs))
      for i, inx in enumerate(np.random.randint(0, len_memory, size = min(batch_size, len_memory))):
          current_state, action, reward, next_state = self.memory[inx][0]
          game_over = self.memory[inx][1]

          inputs[i] = current_state
          targets[i] = model.predict(np.array(current_state).reshape(1, 1, num_inputs))[0]

          if game_over:
              targets[i][action] = reward
          else:
              targets[i][action] = np.argmax(reward + self.discount * model.predict(np.array(next_state).reshape(1, 1, num_inputs))[0][0])

      return inputs, targets

  def train(self, training_epochs = 100, target_change = None):
    if target_change == None:
      target_change = training_epochs
    target_model = keras.models.clone_model(self.model)
    rewards = []
    j = 0
    for i in range(training_epochs):
      j = 0
      game_over = False
      epsilon = self.epsilon
      self.env.reset()
      current_state = np.zeros(self.input_shape).reshape(1, 1, self.input_shape)
      total_reward = 0
      while not game_over:
        if j == target_change:
          target_model = self.model
          j = 0
        if random.random() > epsilon:
          action = random.randint(0, self.output_shape - 1)
        else:
          l = self.model.predict(current_state)[0][0]
          action = np.argmax(l)
        next_state, reward, game_over, info = self.env.step(action)
        next_state = np.array([next_state]).reshape(1, 1, self.input_shape)
        self.remember(current_state, action, reward, next_state, game_over)
        inputs, outputs = self.get_batch(10, target_model)
        self.model.fit(inputs, outputs, epochs = 1, verbose = 0)

        total_reward += reward

        epsilon = epsilon - self.epsilon_decay_rate
        self.epsilon_decay_rate = self.epsilon_decay_rate + self.epsilon_deacy_rate_decay
        current_state = next_state
        j += 1
        print('\rEpisode ' + str(i)+ ' - total reward:' + str(total_reward), 'moves:' + str(j),  end = '')
      rewards.append(total_reward)
      self.rewards = rewards
      plt.plot(rewards)
      plt.show()

In [ ]:
engine = Engine(2000, 'FrozenLake-v0', 0.001)
engine.train(training_epochs=1000, target_change = 20)